# Objective 1b: LDA topics on the r/jobs sample

Fits LDA (latent Dirichlet allocation) to the same 10,000-comment r/jobs sample as the K-means analysis in `2_topic_discovery.ipynb`, as a comparison method. Unlike K-means, LDA gives every comment a mix of topics, which suits short comments that touch several themes.

Pipeline: Part 2's cleaned text → word counts on Part 2's vocabulary rules plus domain stop words → LDA with K = 10 → topic profiles → monthly topic shares → comparison with the K-means clusters.

Settings (chosen 2026-10-31):
- **Domain stop words.** LDA works on raw word counts, so words used everywhere on r/jobs (job, work, company, time, …) filled every topic in a first run. Part 2's `DOMAIN_STOP_WORDS` are removed, and its `PROTECTED_PHRASES` (job market, laid off, entry level, …) are kept as single tokens. The vocabulary therefore differs from the K-means one.
- **K = 10**, chosen from K = 8–20 (section 3).
- **Soft monthly shares** are the main trend measure: each month, the average topic weight over its comments. Which single topic a comment "belongs to" changes a lot between random seeds, so hard assignments are only a secondary view.

To analyse existing model outputs without fitting, start at section 9 in a fresh kernel. The section 3 code cell can also run on its own to plot the recorded K comparison.

## 1. Setup

In [ ]:
# Add the repo root (and scripts/, for Part 2's word lists) to Python's search path
import sys
sys.path.append('..')
sys.path.append('../scripts')

from functions.topics import vectorize_comments, verified_coverage
from functions.topic_reporting import topic_trend_diagnostics
from functions.lda_topics import count_matrix, topic_coherence, match_topics, soft_monthly_shares
from compare_topic_models import DOMAIN_STOP_WORDS, PROTECTED_PHRASES

import os
import json
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

DATA_DIR = Path('../dataset/monthly_Topics')
OUT_DIR = DATA_DIR / 'lda'
FIG_DIR = OUT_DIR / 'figures'
K = 10
SEED, CHECK_SEED = 42, 7
MAX_ITER = 200
RUN_K_DIAGNOSTICS = False  # refits K = 8-20 with two seeds (~15 min); results are recorded in section 3
EARLY_MONTHS = ['2026-01', '2026-02', '2026-03']
LATE_MONTHS = ['2026-07', '2026-08', '2026-09']

# same conversational fillers as notebook 2
EXTRA_TOPIC_STOP_WORDS = [
    'just', 'like', 'really', 'actually', 'probably', 'maybe', 'yes', 'yeah',
    'lol', 'sounds', 'know', 'think', 'people', 'good', 'need', 'want',
    'going', 'got', 'did', 'does', 'said', 'say', 'don', 've', 'll', 'gif', 'giphy',
]

# named on 2026-10-31 from the top words and examples in section 5;
# 'Mixed:' marks low-coherence topics without a single theme
TOPIC_LABELS = {
    0: 'Mixed: workplace, business and services',
    1: 'Job-search difficulty, hiring process and AI',
    2: 'Offers, resigning and changing jobs',
    3: 'Pay, hours, experience and degrees',
    4: 'PTO, leave and bosses',
    5: 'Mixed: personal stories, school and tests',
    6: 'Applications, recruiters, background checks and scams',
    7: 'Mixed: life advice and skills',
    8: 'Advice, good wishes and job-market comments',
    9: 'Economy: wages, cost of living, unemployment and AI',
}

## 2. Data and word counts
Reads Part 2's files from `dataset/monthly_Topics/`. Part 2's `vectorize_comments` builds the vocabulary with its usual rules (moderation notices removed, same stop words) plus the domain stop words and protected phrases, and `count_matrix` turns that vocabulary into the raw counts LDA needs. Comments left with no words are not modelled; they stay in the export as unassigned.

In [ ]:
read = dict(dtype={'comment_id': str}, keep_default_na=False)
corpus = pd.read_csv(DATA_DIR / 'comments_clean.csv', **read)
kmeans = pd.read_csv(DATA_DIR / 'comment_topics.csv', **read)
manifest = json.loads((DATA_DIR / 'sampling_manifest.json').read_text(encoding='utf-8'))
coverage = verified_coverage(DATA_DIR / 'monthly_coverage.csv', manifest['sha256']['monthly_coverage.csv'])

features = vectorize_comments(
    corpus, min_df=3, max_df=0.90, max_features=10000,
    extra_stop_words=EXTRA_TOPIC_STOP_WORDS + DOMAIN_STOP_WORDS, keep_words=['off'],
    exclude_moderation=True, protected_phrases=PROTECTED_PHRASES,
)
docs = features['model_df']
counts, counter = count_matrix(features, docs['text_topic'], PROTECTED_PHRASES)
terms = counter.get_feature_names_out()
n_words = np.asarray(counts.sum(axis=1)).ravel()

kmeans_assigned = set(kmeans.loc[kmeans['topic_status'] == 'assigned', 'comment_id'])
print(f'{len(corpus):,} comments, {len(docs):,} modelled; all within the {len(kmeans_assigned):,} K-means comments:',
      set(docs['comment_id']) <= kmeans_assigned)
print(features['excluded'].groupby('reason').size().to_string())
print(f'Vocabulary: {len(terms):,} terms; median {np.median(n_words):.0f} counted words per comment')

## 3. Choosing K
Results of the K comparison run on 2026-10-31 (seeds 42 and 7, up to 200 iterations). Set `RUN_K_DIAGNOSTICS = True` to recompute them (about 15 minutes).

| K | NPMI mean | NPMI min | Topic diversity | Topic stability, mean / min | Comment ARI |
|---|---|---|---|---|---|
| 8 | 0.107 | 0.036 | 0.85 | 0.62 / 0.53 | 0.02 |
| **10** | **0.113** | **0.060** | 0.83 | 0.54 / 0.39 | 0.02 |
| 12 | 0.106 | 0.018 | 0.82 | 0.53 / 0.32 | 0.02 |
| 15 | 0.092 | −0.053 | 0.77 | 0.49 / 0.28 | 0.02 |
| 18 | 0.079 | −0.117 | 0.78 | 0.46 / 0.30 | 0.01 |
| 20 | 0.051 | −0.186 | 0.77 | 0.49 / 0.33 | 0.01 |

- **NPMI** (coherence): how much more often a topic's top 10 words appear together than by chance; around 0 means unrelated words.
- **Topic diversity**: share of distinct words among all topics' top 10 words; low values mean topics repeat the same words.
- **Topic stability**: cosine similarity of each topic's word distribution with its best match in the seed-7 fit.
- **Comment ARI**: agreement of each comment's top topic between the two seeds (0 = chance).

K = 10 has the highest average coherence among the six tested settings. Some topics still combine several kinds of replies. K = 8 is more stable but its topics are broader, and from K = 15 some topics have negative coherence. Comment ARI is near 0 for every K: comments are short, so single-comment assignments are not reliable and the trends below use soft shares.

In [ ]:
# This cell also runs on its own using the recorded diagnostics.
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Recorded diagnostics from 2026-10-31, rounded to three decimals.
# NPMI is averaged across topics in the seed-42 model.
k_diagnostics = pd.DataFrame({
    'k': [8, 10, 12, 15, 18, 20],
    'npmi_mean': [0.107, 0.113, 0.106, 0.092, 0.079, 0.051],
    'npmi_min': [0.036, 0.060, 0.018, -0.053, -0.117, -0.186],
    'topic_diversity': [0.85, 0.83, 0.82, 0.77, 0.78, 0.77],
    'stability_mean': [0.62, 0.54, 0.53, 0.49, 0.46, 0.49],
    'stability_min': [0.53, 0.39, 0.32, 0.28, 0.30, 0.33],
    'comment_ari': [0.02, 0.02, 0.02, 0.02, 0.01, 0.01],
})

if globals().get('RUN_K_DIAGNOSTICS', False):
    rows = []
    for k in [8, 10, 12, 15, 18, 20]:
        fits = [LatentDirichletAllocation(n_components=k, learning_method='batch', max_iter=MAX_ITER,
                                          evaluate_every=10, random_state=s, n_jobs=-1)
                for s in (SEED, CHECK_SEED)]
        thetas = [m.fit_transform(counts) for m in fits]
        k_coherence = topic_coherence(fits[0].components_, counts)
        k_matched = match_topics(fits[0].components_, fits[1].components_)
        top10 = [terms[w.argsort()[::-1][:10]] for w in fits[0].components_]
        rows.append({
            'k': k, 'npmi_mean': k_coherence['npmi'].mean(), 'npmi_min': k_coherence['npmi'].min(),
            'topic_diversity': len(set(np.concatenate(top10))) / (10 * k),
            'stability_mean': k_matched['cosine'].mean(), 'stability_min': k_matched['cosine'].min(),
            'comment_ari': adjusted_rand_score(thetas[0].argmax(1), thetas[1].argmax(1)),
        })
    k_diagnostics = pd.DataFrame(rows)

display(k_diagnostics.round(3))
fig, ax = plt.subplots(figsize=(8, 4), layout='constrained')
bars = ax.bar(k_diagnostics['k'].astype(str), k_diagnostics['npmi_mean'],
              color=['#7450B8' if k == 10 else '#BCC6D2' for k in k_diagnostics['k']])
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set(xlabel='Number of topics', ylabel='Mean NPMI across topics',
       title='Topic coherence across six candidate models', ylim=(0, 0.14))
k_root = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
k_figure_dir = k_root / 'dataset/monthly_Topics/lda/figures'
k_figure_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(k_figure_dir / 'topic_count_coherence.png', dpi=160, bbox_inches='tight')
plt.show()


## 4. Fit LDA (K = 10)
Fits the final model with seed 42, and again with seed 7 to see which topics are stable. Each comment gets a weight for every topic (its weights sum to 1). The two fits take a few minutes.

In [ ]:
def fit_lda(seed):
    model = LatentDirichletAllocation(n_components=K, learning_method='batch', max_iter=MAX_ITER,
                                      evaluate_every=10, random_state=seed, n_jobs=-1)
    return model, model.fit_transform(counts)


lda, theta = fit_lda(SEED)
lda_check, theta_check = fit_lda(CHECK_SEED)
matched = match_topics(lda.components_, lda_check.components_)
print(f'Iterations: {lda.n_iter_} (seed {SEED}), {lda_check.n_iter_} (seed {CHECK_SEED})')
print('Comment-level ARI between seeds:', round(adjusted_rand_score(theta.argmax(1), theta_check.argmax(1)), 3))

## 5. Topic profiles
For each topic: its share (average weight over all modelled comments), top words, coherence, stability across seeds, and the five comments with the highest weight among those with at least 10 counted words (shorter ones are often a single word). The names in `TOPIC_LABELS` (section 1) were chosen from these. They only fit this model: if the data, settings or seed change, rename the topics, since topic numbers are not carried over.

In [ ]:
coherence = topic_coherence(lda.components_, counts)
order = np.argsort(lda.components_, axis=1)[:, ::-1]
profiles = pd.DataFrame({
    'share': theta.mean(axis=0),
    'top_words': [', '.join(terms[row[:12]]) for row in order],
    'npmi': coherence['npmi'],
    'stability': matched.set_index('topic')['cosine'],
}).rename_axis('topic')
profiles.insert(0, 'label', [TOPIC_LABELS.get(t, ', '.join(terms[order[t, :3]])) for t in range(K)])
chart_labels = {t: f'T{t} {label}' for t, label in profiles['label'].items()}

pd.set_option('display.max_colwidth', 120)
profiles.round(3)

In [ ]:
# the five highest-weight comments per topic, among comments with at least 10 counted words
long_enough = np.flatnonzero(n_words >= 10)
examples = []
for t in range(K):
    for i in long_enough[np.argsort(theta[long_enough, t])[::-1][:5]]:
        examples.append({'topic': t, 'weight': round(theta[i, t], 2),
                         'comment_id': docs['comment_id'].iloc[i], 'text': docs['text'].iloc[i][:220]})
examples = pd.DataFrame(examples)
pd.set_option('display.max_colwidth', 220)
examples

## 6. Monthly topic shares
**Soft shares** (main measure): each month, the average topic weight over that month's comments. The seed-7 fit, with its topics matched to seed 42, shows how much the trends depend on the random seed. January–March is compared with July–September; October is partial (sample cut off on 28 October) and marked with `*`. Every month has the same 1,000-comment quota, so shares show relative attention, not discussion volume.

In [ ]:
shares = soft_monthly_shares(theta, docs['month'])
shares_check = soft_monthly_shares(theta_check, docs['month'])[matched['matched_topic'].to_numpy()]
shares_check.columns = matched['topic'].to_numpy()

partial = set(coverage.loc[coverage['partial_month'].astype(str).str.lower() == 'true', 'month'])
month_ticks = [m[5:] + ('*' if m in partial else '') for m in shares.index]


def period_change(monthly):
    early, late = monthly.loc[EARLY_MONTHS].mean(), monthly.loc[LATE_MONTHS].mean()
    return pd.DataFrame({'jan_mar': early, 'jul_sep': late, 'change_pp': (late - early) * 100})


trend = period_change(shares).join(period_change(shares_check)['change_pp'].rename('change_pp_seed7'))
trend['same_direction'] = np.sign(trend['change_pp']) == np.sign(trend['change_pp_seed7'])
trend.insert(0, 'label', profiles['label'])
trend.round(3)

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)

fig, ax = plt.subplots(figsize=(13, 6))
image = ax.imshow(shares.T.to_numpy() * 100, aspect='auto', cmap='Blues')
ax.set_xticks(range(len(month_ticks)), month_ticks)
ax.set_yticks(range(K), [chart_labels[t] for t in range(K)])
for row in range(K):
    for col in range(len(shares)):
        ax.text(col, row, f'{shares.iloc[col, row] * 100:.1f}', ha='center', va='center', fontsize=8)
ax.set(xlabel='2026 UTC month (* partial)', title='LDA soft topic shares per month (%)')
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / 'soft_shares_heatmap.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(20, 7), sharex=True, sharey=True)
x = range(len(shares))
for t, ax in zip(range(K), axes.ravel()):
    ax.plot(x, shares[t] * 100, marker='o', label=f'seed {SEED}')
    ax.plot(x, shares_check[t] * 100, linestyle='--', label=f'seed {CHECK_SEED} (matched)')
    ax.set_title(chart_labels[t], fontsize=9)
    ax.set_xticks(list(x), month_ticks, fontsize=7)
for ax in axes[:, 0]:
    ax.set_ylabel('Soft share (%)')
axes[0, 0].legend(fontsize=7)
plt.tight_layout()
plt.savefig(FIG_DIR / 'soft_shares_by_topic.png', dpi=150)
plt.show()

### Secondary: top topic per comment
Each comment assigned to its highest-weight topic and passed to Part 2's `topic_trend_diagnostics`, so the output has the same form as the K-means trends (including equal weight per post). Given the low comment-level stability, use this only as a cross-check of the soft shares.

In [ ]:
hard = docs.assign(cluster=theta.argmax(axis=1))
_, _, hard_monthly, hard_summary = topic_trend_diagnostics(
    hard, coverage[['month', 'partial_month', 'source_window_complete']], EARLY_MONTHS, LATE_MONTHS,
)
hard_summary.insert(1, 'label', hard_summary['cluster'].map(profiles['label']))
hard_summary[['cluster', 'label', 'n_comments', 'first3_share', 'last3_share', 'change_pp',
              'equal_post_change_pp']].round(3)

## 7. Comparison with K-means
For each K-means cluster (with Part 2's suggested names), the average LDA topic mix of its comments: where its content goes in the LDA model. Rows sum to 100%. NMI compares the K-means clusters with each comment's LDA top topic (0 = unrelated, 1 = the same partition); it inherits the instability of the top topic, so read the broad pattern rather than single cells.

In [ ]:
km = kmeans.set_index('comment_id').loc[docs['comment_id']]
km_cluster = km['cluster'].astype(float).astype(int).to_numpy()
km_names = km.assign(cluster=km_cluster).groupby('cluster')['topic_label'].first()

mix = pd.DataFrame(theta).groupby(km_cluster).mean()
mix.index = [f'C{c} {km_names[c]}' for c in mix.index]
mix.columns = [chart_labels[t] for t in mix.columns]
print('NMI, K-means clusters vs LDA top topics:',
      round(normalized_mutual_info_score(km_cluster, theta.argmax(axis=1)), 3))

fig, ax = plt.subplots(figsize=(14, 9))
image = ax.imshow(mix.to_numpy() * 100, aspect='auto', cmap='Purples')
ax.set_xticks(range(K), mix.columns, rotation=40, ha='right', fontsize=8)
ax.set_yticks(range(len(mix)), mix.index, fontsize=8)
for row in range(len(mix)):
    for col in range(K):
        ax.text(col, row, f'{mix.iloc[row, col] * 100:.0f}', ha='center', va='center', fontsize=7)
ax.set(xlabel='LDA topic', ylabel='K-means cluster', title='Average LDA topic mix of each K-means cluster (%)')
fig.colorbar(image, ax=ax)
plt.tight_layout()
plt.savefig(FIG_DIR / 'kmeans_vs_lda.png', dpi=150)
plt.show()

## 8. Export
Writes to `dataset/monthly_Topics/lda/` (ignored by git):
- `lda_comment_topics.csv`: all 10,000 comments with their status, top topic, its weight and the weight of every topic.
- `lda_topic_profiles.csv`, `lda_monthly_shares.csv` (seed 42 and matched seed 7), `lda_trend_summary.csv`, `kmeans_lda_mix.csv`.
- `lda_run.json` with the settings and input hash, and the figures above in `figures/`.

In [ ]:
weights = pd.DataFrame(theta, columns=[f'topic_{t}' for t in range(K)])
modelled = docs[['comment_id', 'month']].reset_index(drop=True).join(weights)
modelled.insert(2, 'lda_status', 'modelled')
modelled.insert(3, 'top_topic', theta.argmax(axis=1))
modelled.insert(4, 'top_weight', theta.max(axis=1))
unmodelled = features['excluded'].rename(columns={'reason': 'lda_status'})
unmodelled['lda_status'] = unmodelled['lda_status'].replace({'zero_tfidf': 'no_words_left'})
comment_topics = pd.concat([modelled, unmodelled], ignore_index=True)
comment_topics['top_topic'] = comment_topics['top_topic'].astype('Int64')
assert len(comment_topics) == len(corpus) and comment_topics['comment_id'].is_unique

monthly = pd.concat({'seed_42': shares.stack(), 'seed_7_matched': shares_check.stack()}, axis=1)
monthly = monthly.rename_axis(['month', 'topic']).reset_index()

comment_topics.to_csv(OUT_DIR / 'lda_comment_topics.csv', index=False)
profiles.to_csv(OUT_DIR / 'lda_topic_profiles.csv')
monthly.to_csv(OUT_DIR / 'lda_monthly_shares.csv', index=False)
trend.to_csv(OUT_DIR / 'lda_trend_summary.csv')
mix.to_csv(OUT_DIR / 'kmeans_lda_mix.csv')
run = {
    'k': K, 'seed': SEED, 'check_seed': CHECK_SEED, 'max_iter': MAX_ITER,
    'iterations': [int(lda.n_iter_), int(lda_check.n_iter_)],
    'modelled_comments': len(docs), 'vocabulary_size': len(terms),
    'stop_words': 'sklearn English + notebook 2 fillers + compare_topic_models.DOMAIN_STOP_WORDS; keep "off"',
    'protected_phrases': PROTECTED_PHRASES, 'topic_labels': TOPIC_LABELS,
    'input_sha256': hashlib.sha256((DATA_DIR / 'comments_clean.csv').read_bytes()).hexdigest(),
}
(OUT_DIR / 'lda_run.json').write_text(json.dumps(run, indent=2), encoding='utf-8')
print('Saved to', OUT_DIR)

## 9. Additional analyses of the fitted LDA model

This section uses the saved comment-level topic weights to examine the fitted
model from four perspectives:

1. the overall distribution of topic weights;
2. the relationship between the LDA topics and the K-means clusters;
3. changes in topic weights between the first and third quarters; and
4. the sensitivity of these changes to popular posts and the random seed.

These analyses use the outputs generated in Section 8, so the LDA models do not
need to be fitted again. If the input data or model settings change, rerun the
earlier sections and regenerate the exports before running this section.

The generated tables are saved under `lda/analysis/`, and the figures are saved
under `lda/figures/`.

In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

analysis_root = Path.cwd()
if analysis_root.name == 'notebook':
    analysis_root = analysis_root.parent
topic_dir = analysis_root / 'dataset/monthly_Topics'
lda_dir = topic_dir / 'lda'
analysis_dir = lda_dir / 'analysis'
analysis_fig_dir = lda_dir / 'figures'
analysis_dir.mkdir(parents=True, exist_ok=True)
analysis_fig_dir.mkdir(parents=True, exist_ok=True)

lda_metadata = json.loads((lda_dir / 'lda_run.json').read_text(encoding='utf-8'))
input_hash = hashlib.sha256((topic_dir / 'comments_clean.csv').read_bytes()).hexdigest()
assert input_hash == lda_metadata['input_sha256'], 'Sample changed; regenerate the LDA exports.'
assert lda_metadata['k'] == 10 and lda_metadata['seed'] == 42
assert lda_metadata['check_seed'] == 7

analysis_corpus = pd.read_csv(topic_dir / 'comments_clean.csv',
                              dtype={'comment_id': str, 'link_id': str}, keep_default_na=False)
analysis_weights = pd.read_csv(lda_dir / 'lda_comment_topics.csv', dtype={'comment_id': str})
analysis_profiles = pd.read_csv(lda_dir / 'lda_topic_profiles.csv').set_index('topic')
analysis_monthly = pd.read_csv(lda_dir / 'lda_monthly_shares.csv')
analysis_kmeans = pd.read_csv(topic_dir / 'comment_topics.csv', dtype={'comment_id': str})
topic_columns = [f'topic_{t}' for t in range(10)]
assert analysis_weights['comment_id'].is_unique
assert set(analysis_weights['comment_id']) == set(analysis_corpus['comment_id'])
analysis_docs = analysis_weights.loc[analysis_weights['lda_status'].eq('modelled')].merge(
    analysis_corpus[['comment_id', 'month', 'link_id', 'text']],
    on=['comment_id', 'month'], how='left', validate='one_to_one', indicator=True,
)
assert analysis_docs['_merge'].eq('both').all()
assert analysis_docs['link_id'].notna().all() and analysis_docs['link_id'].ne('').all()
assert len(analysis_docs) == lda_metadata['modelled_comments']
assert np.isfinite(analysis_docs[topic_columns]).all().all()
assert (analysis_docs[topic_columns] >= 0).all().all()
np.testing.assert_allclose(analysis_docs[topic_columns].sum(axis=1), 1, atol=1e-8)

topic_names = {
    0: 'Workplace & services', 1: 'Job search & AI', 2: 'Changing jobs',
    3: 'Pay & qualifications', 4: 'Leave & managers', 5: 'Personal stories',
    6: 'Hiring process', 7: 'Life advice', 8: 'Advice & encouragement',
    9: 'Economy & unemployment',
}
topic_colors = {t: '#92A4B7' for t in range(10)}
topic_colors.update({t: '#BCC6D2' for t in [0, 5, 7]})
topic_colors.update({2: '#087F72', 3: '#3D79A5', 6: '#7450B8', 9: '#BB6E2B'})
mixed_topics = [0, 5, 7]
early_months = ['2026-01', '2026-02', '2026-03']
late_months = ['2026-07', '2026-08', '2026-09']


def save_analysis_figure(fig, filename):
    fig.savefig(analysis_fig_dir / filename, dpi=160, bbox_inches='tight')
    plt.show()


model_counts = pd.Series({
    'sample_comments': len(analysis_corpus),
    'modelled_comments': len(analysis_docs),
    'excluded_comments': len(analysis_corpus) - len(analysis_docs),
    'vocabulary_size': lda_metadata['vocabulary_size'],
    'topics': lda_metadata['k'],
    'seed_42_iterations': lda_metadata['iterations'][0],
    'seed_7_iterations': lda_metadata['iterations'][1],
}, name='count')
display(model_counts.to_frame())
display(analysis_corpus.groupby('month').size().rename('sample_comments').to_frame())

### 9.1 Comparing a single assignment with topic weights

To illustrate the difference between the two topic representations, retrieve one
comment by ID and compare its K-means cluster with its LDA topic weights. K-means
assigns the complete comment to one cluster, whereas LDA distributes its weight
across all ten topics.

The chart displays the two largest interpretable topic weights separately and
combines the remaining weights as `Other topics`. This example explains how a
comment with more than one theme contributes to the monthly LDA averages.

In [ ]:
example_id = 'p882nzc'
example_comment = analysis_docs.set_index('comment_id').loc[example_id]
example_cluster = analysis_kmeans.set_index('comment_id').loc[example_id]
print(example_comment['text'])
print(f"K-means C{int(example_cluster['cluster'])}: {example_cluster['topic_label']}")
example_mix = pd.Series({
    'T2 Changing jobs': example_comment['topic_2'] * 100,
    'T3 Pay & qualifications': example_comment['topic_3'] * 100,
    'Other topics': (1 - example_comment['topic_2'] - example_comment['topic_3']) * 100,
}, name='weight_pct')
display(example_mix.to_frame().round(3))
fig, ax = plt.subplots(figsize=(10, 2.5), layout='constrained')
left = 0
for (label, value), color in zip(example_mix.items(), ['#087F72', '#3D79A5', '#DCE3EA']):
    ax.barh(0, value, left=left, color=color, label=f'{label}: {value:.1f}%')
    left += value
ax.set(xlim=(0, 100), yticks=[], xlabel='Topic weight (%)',
       title='One comment contributes to multiple topics')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.3), ncol=3)
save_analysis_figure(fig, 'comment_topic_mix.png')
example_mix.to_csv(analysis_dir / 'example_topic_weights.csv')

### 9.2 Overall topic composition

Calculate each topic's overall share as its mean weight across all modelled
comments. Because every comment's topic weights sum to one, the ten mean weights
also sum to one. This provides a common baseline for comparing topic prevalence
and for interpreting the cluster-level averages in the next section.

The table also includes each topic's coherence, matched similarity across random
seeds and three keyword cues taken from its saved top-word list. T0, T5 and T7
retain the `Mixed` designation assigned during topic interpretation; their
combined share is calculated separately to describe how much weight falls on
these broader profiles.

In [ ]:
overall_weights = analysis_docs[topic_columns].mean()
overall_weights.index = range(10)
np.testing.assert_allclose(overall_weights, analysis_profiles.loc[range(10), 'share'])
keyword_cues = {
    0: ['office', 'business', 'customer'], 1: ['ai', 'interview', 'hiring'],
    2: ['offer', 'notice', 'new_job'], 3: ['experience', 'hours', 'pay'],
    4: ['boss', 'pto', 'leave'], 5: ['manager', 'school', 'test'],
    6: ['resume', 'background', 'recruiter'], 7: ['skills', 'world', 'family'],
    8: ['best', 'luck', 'advice'], 9: ['income', 'living', 'unemployment'],
}
for topic, cues in keyword_cues.items():
    assert set(cues) <= set(analysis_profiles.loc[topic, 'top_words'].split(', '))
topic_overview = pd.DataFrame({
    'name': pd.Series(topic_names), 'share_pct': overall_weights * 100,
    'keyword_cues': pd.Series({t: ', '.join(cues) for t, cues in keyword_cues.items()}),
    'npmi': analysis_profiles['npmi'], 'matched_similarity': analysis_profiles['stability'],
}).rename_axis('topic')
display(topic_overview.round(3))
print(f'Topic shares range from {overall_weights.min():.1%} to {overall_weights.max():.1%}.')
print(f'Mixed-topic weight: {overall_weights.loc[mixed_topics].sum():.2%}.')

ordered_topics = overall_weights.sort_values(ascending=False).index
fig, ax = plt.subplots(figsize=(12, 6), layout='constrained')
bars = ax.barh(range(10), overall_weights.loc[ordered_topics] * 100,
               color=[topic_colors[t] for t in ordered_topics])
ax.bar_label(bars, fmt='%.1f%%', padding=3)
ax.set_yticks(range(10), [f'T{t} {topic_names[t]}' for t in ordered_topics])
for y, topic in enumerate(ordered_topics):
    ax.text(15, y, ', '.join(keyword_cues[topic]), va='center', fontsize=10)
ax.invert_yaxis()
ax.set(xlim=(0, 23), xlabel='Mean topic weight (%)',
       title='Ten topics with similar overall shares')
ax.set_xticks([0, 4, 8, 12])
save_analysis_figure(fig, 'topic_composition.png')
topic_overview.to_csv(analysis_dir / 'topic_composition.csv')

### 9.3 Content correspondence with K-means

Join the K-means and LDA outputs by `comment_id`. For each K-means cluster,
average the ten LDA topic weights of its comments. Each row therefore describes
the internal topic composition of one K-means cluster and sums to 100%.

Add the overall topic weights as a baseline. An enrichment ratio divides a
cluster's topic weight by the corresponding overall weight, making it possible
to distinguish a cluster-specific concentration from a topic that is common
throughout the corpus.

The selected heatmap shows four clusters used for detailed interpretation, while
the two full heatmaps retain all 18 clusters. For the AI cluster, T1 and T9 are
also added together to examine how its weight is distributed across job-search
and economic topic profiles. This combined value is a topic-weight total, not a
percentage of comments about a specific AI effect.

In [ ]:
cross_model = analysis_docs[['comment_id'] + topic_columns].merge(
    analysis_kmeans[['comment_id', 'cluster']], on='comment_id',
    how='left', validate='one_to_one', indicator=True,
)
assert cross_model['_merge'].eq('both').all() and cross_model['cluster'].notna().all()
cross_model['cluster'] = cross_model['cluster'].astype(int)
cluster_mix = cross_model.groupby('cluster')[topic_columns].mean() * 100
cluster_mix.columns = range(10)
np.testing.assert_allclose(cluster_mix.sum(axis=1), 100)
selected_mix = cluster_mix.loc[[9, 11, 17, 7]].copy()
selected_mix.index = ['C9 Changing jobs', 'C11 Qualifications', 'C17 AI cluster', 'C7 General replies']
selected_mix.loc['All comments'] = overall_weights * 100
display(selected_mix.round(1))
enrichment = pd.Series({
    'C9 T2 / overall T2': cluster_mix.loc[9, 2] / (overall_weights[2] * 100),
    'C11 T3 / overall T3': cluster_mix.loc[11, 3] / (overall_weights[3] * 100),
    'C17 T1 weight (%)': cluster_mix.loc[17, 1],
    'C17 T9 weight (%)': cluster_mix.loc[17, 9],
    'C17 T1 + T9 weight (%)': cluster_mix.loc[17, [1, 9]].sum(),
}, name='value')
display(enrichment.round(3).to_frame())


def plot_content_mix(table, title, filename, decimals=1):
    fig, ax = plt.subplots(figsize=(14, 0.52 * len(table) + 3), layout='constrained')
    image = ax.imshow(table, aspect='auto', cmap='Blues', vmin=0, vmax=35)
    ax.set_xticks(range(10), [f'T{t} {topic_names[t]}' for t in range(10)],
                  rotation=35, ha='right', fontsize=9)
    ax.set_yticks(range(len(table)), table.index)
    for row in range(len(table)):
        for column in range(10):
            value = table.iloc[row, column]
            ax.text(column, row, f'{value:.{decimals}f}', ha='center', va='center',
                    color='white' if value > 23 else '#182C43', fontsize=9)
    ax.set_title(title)
    fig.colorbar(image, ax=ax, label='Mean topic weight (%)', shrink=0.7)
    save_analysis_figure(fig, filename)


plot_content_mix(selected_mix, 'K-means clusters and their LDA content', 'selected_content_heatmap.png')
for first, last in [(0, 8), (9, 17)]:
    block = cluster_mix.loc[first:last].rename(index=lambda cluster: f'C{cluster}')
    plot_content_mix(block, f'Full content map, C{first}–C{last}',
                     f'full_content_heatmap_c{first}_c{last}.png', decimals=0)
cluster_mix.to_csv(analysis_dir / 'cluster_topic_weights_pct.csv')
enrichment.to_csv(analysis_dir / 'cluster_enrichment.csv')

### 9.4 Quarterly changes and their scale

Average the monthly topic weights for January–March and July–September. Giving
each month equal weight keeps the comparison consistent with the monthly sampling
design. October is excluded because it is a partial month.

Report change in two forms. Percentage-point change is the Q3 mean minus the Q1
mean. Relative change divides this difference by the Q1 mean. The first chart
retains all ten topics so that the detailed cases can be viewed in context.

The detailed comparison focuses on T2, T6 and T9 because their topic profiles
have relatively specific interpretations. Topics with broader or mixed profiles
remain in the all-topic chart but are not used as the main examples. The T2–T9
gap is calculated in each quarter to compare their distance on the same
percentage-point scale.

In [ ]:
monthly_weights = analysis_docs.groupby('month')[topic_columns].mean()
monthly_weights.columns = range(10)
exported_monthly = analysis_monthly.pivot(index='month', columns='topic', values='seed_42')
np.testing.assert_allclose(monthly_weights, exported_monthly.loc[monthly_weights.index])
q1_weights = monthly_weights.loc[early_months].mean() * 100
q3_weights = monthly_weights.loc[late_months].mean() * 100
quarterly_topics = pd.DataFrame({
    'name': pd.Series(topic_names), 'q1_share_pct': q1_weights, 'q3_share_pct': q3_weights,
    'change_pp': q3_weights - q1_weights,
    'relative_change_pct': (q3_weights / q1_weights - 1) * 100,
}).rename_axis('topic')
display(quarterly_topics.round(3))
gap_q1 = q1_weights[2] - q1_weights[9]
gap_q3 = q3_weights[2] - q3_weights[9]
topic_gap = pd.Series({'q1_gap_pp': gap_q1, 'q3_gap_pp': gap_q3,
                       'gap_multiple': gap_q3 / gap_q1}, name='value')
display(topic_gap.round(3).to_frame())

change_order = quarterly_topics['change_pp'].sort_values(ascending=False).index
fig, axes = plt.subplots(1, 2, figsize=(14, 6), layout='constrained',
                         gridspec_kw={'width_ratios': [1.6, 1]})
bars = axes[0].barh(range(10), quarterly_topics.loc[change_order, 'change_pp'],
                    color=[topic_colors[t] for t in change_order])
axes[0].bar_label(bars, fmt='%+.2f', padding=4)
axes[0].set_yticks(range(10), [f'T{t} {topic_names[t]}' for t in change_order])
axes[0].invert_yaxis()
axes[0].axvline(0, color='grey', linewidth=0.8)
axes[0].set(xlabel='Q3 minus Q1 (percentage points)', xlim=(-2.2, 1.7), title='All topic changes')
for topic in [2, 9]:
    axes[1].plot([0, 1], [q1_weights[topic], q3_weights[topic]], '-o',
                  color=topic_colors[topic], label=f'T{topic} {topic_names[topic]}')
    for x, value in enumerate([q1_weights[topic], q3_weights[topic]]):
        axes[1].annotate(f'{value:.1f}%', (x, value), xytext=(0, 9),
                          textcoords='offset points', ha='center')
axes[1].set(xticks=[0, 1], xticklabels=['Q1', 'Q3'], ylabel='Mean topic weight (%)',
             ylim=(0, 14), xlim=(-0.15, 1.15), title=f'T2–T9 gap expands {gap_q3 / gap_q1:.1f}×')
axes[1].legend(loc='lower left', fontsize=9)
save_analysis_figure(fig, 'quarterly_topic_changes.png')
quarterly_topics.to_csv(analysis_dir / 'quarterly_topic_changes.csv')
topic_gap.to_csv(analysis_dir / 'quarterly_topic_gap.csv')

### 9.5 Sensitivity to popular posts

Check whether the quarterly changes depend heavily on a small number of posts.
For each topic and month, rank posts by the sum of topic weights across their
sampled comments. Remove the top three posts for that topic and recompute the
mean topic weight from the remaining comments. Different topics can therefore
have different TOP3 posts. This ranking measures topic contribution within the
sample rather than Reddit votes or engagement.

As a second check, first average comment weights within each post-month and then
give every post the same weight in that month. This reduces the influence of
posts that contribute many sampled comments. Both checks continue to give each
month equal weight in the quarterly comparison.

Attenuation is calculated as
`100 × (1 − change_without_top3 / original_change)`. It describes how much the
estimated quarterly change shrinks after TOP3 removal; it is not a causal share
attributable to those posts.

In [ ]:
post_topic_totals = analysis_docs.groupby(['month', 'link_id'])[topic_columns].sum()
post_topic_means = analysis_docs.groupby(['month', 'link_id'])[topic_columns].mean()
equal_post_monthly = post_topic_means.groupby('month').mean()
equal_post_monthly.columns = range(10)
top_post_rows = []
removed_monthly_rows = []
for topic in range(10):
    column = f'topic_{topic}'
    for month, month_docs in analysis_docs.groupby('month'):
        ranked = post_topic_totals.loc[month, column].sort_values(ascending=False, kind='stable')
        assert len(ranked) > 3, f'Too few posts in {month} for TOP3 removal.'
        top_ids = ranked.head(3).index
        remaining = month_docs.loc[~month_docs['link_id'].isin(top_ids)]
        removed_monthly_rows.append({
            'month': month, 'topic': topic,
            'without_top3': remaining[column].mean(),
            'removed_comments': len(month_docs) - len(remaining),
            'remaining_comments': len(remaining),
        })
        for rank, post_id in enumerate(top_ids, start=1):
            top_post_rows.append({'month': month, 'topic': topic, 'rank': rank,
                                  'link_id': post_id, 'summed_topic_weight': ranked[post_id]})
removed_monthly = pd.DataFrame(removed_monthly_rows)
without_top3_monthly = removed_monthly.pivot(index='month', columns='topic', values='without_top3')


def quarter_difference(monthly_table):
    return (monthly_table.loc[late_months].mean() - monthly_table.loc[early_months].mean()) * 100


post_sensitivity = pd.DataFrame({
    'original_pp': quarter_difference(monthly_weights),
    'without_top3_pp': quarter_difference(without_top3_monthly),
    'equal_posts_pp': quarter_difference(equal_post_monthly),
}).rename_axis('topic')
nonzero_change = post_sensitivity['original_pp'].mask(post_sensitivity['original_pp'].abs() < 1e-12)
post_sensitivity['attenuation_pct'] = (1 - post_sensitivity['without_top3_pp'] / nonzero_change) * 100
display(post_sensitivity.loc[[2, 6, 9]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), layout='constrained',
                         gridspec_kw={'width_ratios': [2, 1]})
case_topics = [2, 6, 9]
for column, marker, offset, label in [
    ('original_pp', 'o', -0.18, 'Original'),
    ('without_top3_pp', 's', 0, 'Remove TOP3'),
    ('equal_posts_pp', 'D', 0.18, 'Equal posts'),
]:
    values = post_sensitivity.loc[case_topics, column]
    axes[0].scatter(values, np.arange(3) + offset, marker=marker, s=65,
                     c=[topic_colors[t] for t in case_topics], label=label)
    for y, value in enumerate(values):
        axes[0].annotate(f'{value:+.2f}', (value, y + offset), xytext=(7, 0),
                          textcoords='offset points', va='center', fontsize=9)
axes[0].axvline(0, color='grey', linewidth=0.8)
axes[0].set_yticks(range(3), [f'T{t} {topic_names[t]}' for t in case_topics])
axes[0].invert_yaxis()
axes[0].set(xlim=(-2.1, 1.65), xlabel='Q3 minus Q1 (percentage points)',
             title='Quarterly change under different post weights')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.17), ncol=3, fontsize=9)
attenuation_order = [6, 2, 9]
bars = axes[1].barh(range(3), post_sensitivity.loc[attenuation_order, 'attenuation_pct'],
                    color=[topic_colors[t] for t in attenuation_order])
axes[1].bar_label(bars, fmt='%.0f%%', padding=4)
axes[1].set_yticks(range(3), [f'T{t}' for t in attenuation_order])
axes[1].invert_yaxis()
axes[1].set(xlim=(0, 80), xlabel='Reduction in change magnitude (%)', title='After removing TOP3')
save_analysis_figure(fig, 'post_weight_sensitivity.png')
post_sensitivity.to_csv(analysis_dir / 'post_weight_sensitivity.csv')
removed_monthly.to_csv(analysis_dir / 'top3_removed_monthly.csv', index=False)
pd.DataFrame(top_post_rows).to_csv(analysis_dir / 'top3_posts_by_topic_month.csv', index=False)

### 9.6 Sensitivity to the random seed

LDA can produce different topic numbers and word distributions under a different
initialisation. Before comparing the two fits, match each seed-7 topic to the
seed-42 topic with the most similar word distribution.

For T2, T6 and T9, compare the Q3-minus-Q1 change from the main fit with the
change from the matched seed-7 fit. Report the cosine similarity used in the
matching alongside each comparison. This checks whether the direction and
magnitude of a trend depend on the random initialisation; the cosine similarity
is a topic-matching measure rather than a significance test.

In [ ]:
matched_monthly = analysis_monthly.pivot(index='month', columns='topic', values='seed_7_matched')
seed_sensitivity = pd.DataFrame({
    'seed_42_pp': quarter_difference(monthly_weights),
    'seed_7_pp': quarter_difference(matched_monthly),
    'matched_cosine': analysis_profiles['stability'],
}).rename_axis('topic')
display(seed_sensitivity.loc[case_topics].round(3))
fig, ax = plt.subplots(figsize=(10, 4), layout='constrained')
for column, marker, offset, label in [
    ('seed_42_pp', 'o', -0.10, 'Seed 42'), ('seed_7_pp', 's', 0.10, 'Seed 7, matched'),
]:
    values = seed_sensitivity.loc[case_topics, column]
    ax.scatter(values, np.arange(3) + offset, marker=marker, s=70,
               c=[topic_colors[t] for t in case_topics], label=label)
    for y, value in enumerate(values):
        ax.annotate(f'{value:+.2f}', (value, y + offset), xytext=(7, 0),
                    textcoords='offset points', va='center')
ax.axvline(0, color='grey', linewidth=0.8)
ax.set_yticks(range(3), [f'T{t} {topic_names[t]} (match {seed_sensitivity.loc[t, "matched_cosine"]:.2f})'
                        for t in case_topics])
ax.invert_yaxis()
ax.set(xlim=(-2.1, 1.8), xlabel='Q3 minus Q1 (percentage points)', title='Random-seed sensitivity')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2)
save_analysis_figure(fig, 'random_seed_sensitivity.png')
seed_sensitivity.to_csv(analysis_dir / 'random_seed_sensitivity.csv')

## 10. Notes
- LDA is a comparison with the proposal's K-means, not a replacement. Its vocabulary differs (domain stop words), and the comments left with no words are not modelled.
- Comments are short, so which topic a single comment belongs to is unstable between seeds. Read trends from the soft shares and check the seed-7 column before describing a change.
- Every month has the same 1,000-comment quota, so shares show relative attention, not volume. October is partial.
- Topic names are the group's reading of the top words and example comments, not ground truth.